# Data Preparation - Red Flag Detection

Questo notebook documenta e applica interattivamente la *pipeline* di Data Preparation (pulizia e trasformazione testuale) sul corpus grezzo `Suicide Watch`, al fine di generare feature adeguate per l'addestramento dei modelli NLP.

### Decisioni Architetturali (Pipeline):
1. **Noise Removal:** Eliminazione tramite RegEx di URL esterni, tag HTML passivi e formattazioni anomale (es. occorrenze esasperate di ritorni a capo `\n`), per uniformare l'input visivo a quello di una textarea da diario.
2. **Case-Folding & Text Normalization:** Conversione totale dei caratteri in *lowercasing* per non sdoppiare il mapping vettoriale dei token.
3. **Conservazione Punteggiatura Emotiva:** I marcatori di interpunzione di fine frase (`.`, `,`, `!`, `?`) sono preservati per fornire al layer di *Attention* del Transformer informazioni essenziali sull'enfasi e l'ansia dell'utente.
4. **Stop-Words & Lemmatization:** Tramite `NLTK`, sono stati defalcati gli elementi grammaticali irrilevanti (articoli, pronomi), **con l'eccezione critica della whitelist delle negazioni** (`not`, `never`, `isn't`...). Contestualmente, il `WordNetLemmatizer` è applicato per aggregare forme coniugate della stessa radice.
5. **Filtro Dimensionale (Max 2000 chr):** Per simulare l'ambiente di immissione reale dei diari clinici, i post testuali più lunghi di 2000 caratteri vengono eliminati integralmente. Questo approccio è decisamente preferibile al semplice 'troncamento' della stringa, poiché impedisce di perdere porzioni fatidiche del testo, spesso relegate proprio all'epilogo per il fenomeno psicologico del Recency Bias.

***Nota di Integrazione Multilingua:** I log su cui il sistema opererà in produzione saranno prevalentemente scritti in italiano (SINTON-IA Campania). Per arginare il *Language Gap* senza incorrere nei bias della *Machine Translation*, questo dataset non subirà traduzioni pre-training: l'architettura farà affidamento al trasferimento logico Zero-Shot tramite l'utilizzo a valle di *Multilingual Representation Transformers* (es. XLM-R / mBERT)*.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Setup
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Caricamento dati raw...")
df = pd.read_csv('../data/raw/Suicide_Detection.csv')
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)
df.head(2)

### Step 1: Rimozione URLs e Formattazioni Anomale (Newlines)

In [ ]:
def remove_urls_and_newlines(text):
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'\n+', ' ', text)
    return text

df['step1_text'] = df['text'].apply(remove_urls_and_newlines)
df[['text', 'step1_text']].head(2)

### Step 2: Lowercasing e Pulizia Caratteri Speciali (Punteggiatura consentita)

In [ ]:
def clean_chars_and_lower(text):
    text = text.lower()
    # Manteniamo . , ! ?
    text = re.sub(r'[^a-zA-Z0-9\s\.,!\?]', '', text)
    # Stacchiamo la punteggiatura dalle parole per la successiva lemmatizzazione
    text = re.sub(r'([\.,!\?])', r' \1 ', text)
    # Rimuoviamo doppi spazi accidentali
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['step2_text'] = df['step1_text'].apply(clean_chars_and_lower)
df[['step1_text', 'step2_text']].head(2)

### Step 3: Stopword Whitelist e Lemmatization
Prepariamo la whitelist delle negazioni da proteggere e applichiamo il lemmatizzatore ai restanti lemmi.

In [ ]:
stop_words = set(stopwords.words('english'))
# Selettivita': Salvare le forme negative
words_to_keep = {'not', 'no', 'nor', 'don', "don't", 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't", 'never'}
stop_words = stop_words - words_to_keep

lemmatizer = WordNetLemmatizer()

def remove_stops_and_lemmatize(text):
    words = text.split()
    cleaned_words = []
    for word in words:
        if word in ['.', ',', '!', '?']:
            cleaned_words.append(word)
        elif word not in stop_words:
            cleaned_words.append(lemmatizer.lemmatize(word))
    return ' '.join(cleaned_words)

print("Applicazione Lemma (su CPU richiede ~1 minuto)...")
df['step3_text'] = df['step2_text'].apply(remove_stops_and_lemmatize)
df[['step2_text', 'step3_text']].head(2)

### Step 4: Protezione Architetturale (Max 2000 caratteri) e Recency Bias
Simuliamo i limiti di form del frontend SINTON-IA (max 2000 chr). A differenza della pregressa idea del troncamento indiscriminato, andremo ad **ELIMINARE** interamente le righe eccedenti. La motivazione risiede nell'Effetto Recency psicologico: l'imminenza del rischio di suicidio si cela quasi sempre alla fine dei log testuali lunghi. Un troncamento maschererebbe tale segnale originando pericolosi Falsi-Negativi. Eseguendo un filtro secco e rimuovendo i documenti prolissi, manterremo puro il contesto e valuteremo in seguito eventuali impatti sul bilanciamento delle label.

In [ ]:
# Filtriamo i record vuoti derivati dallo step 3 (length 0)
# ED escludiamo tutte le frasi lunghe oltre il massimale architetturale (length > 2000)
df['char_count'] = df['step3_text'].apply(lambda x: len(str(x)))
df_final = df[(df['char_count'] > 0) & (df['char_count'] <= 2000)].copy()

df_final = df_final[['step3_text', 'class']]
df_final.rename(columns={'step3_text': 'text'}, inplace=True)

print(f"Totale righe finali idonee al salvataggio: {len(df_final)}")
print("\n--- Bilanciamento Naturale Residuo ---")
print(df_final['class'].value_counts(normalize=True))
df_final.head()

### Step 5: Esportazione in Processed

In [ ]:
df_final.to_csv('../data/processed/red_flag_cleaned.csv', index=False)
print("Esportazione terminata. Pronti per HF Hub!")